# Question-1: Image Denoising using Denoising Autoencoder (DAE)
**Dataset:** The Berkeley Segmentation Dataset and Benchmark (BSDS500)

This notebook demonstrates the end-to-end pipeline for training and evaluating a Convolutional Denoising Autoencoder (ConvDAE) on BSDS500 images corrupted with **Gaussian noise** and **Salt-and-Pepper noise**.

### Pipeline Overview:
1. **Dataset Preparation:** BSDS500 download, normalization to $[0, 1]$, and train/val/test splits (200/100/200).
2. **Noise Injection:** Gaussian noise $\mathcal{N}(0, \sigma^2)$ and Salt-and-Pepper impulse noise.
3. **Model Architecture:** Convolutional encoder-bottleneck-decoder with configurable bottleneck capacity.
4. **Training:** Supervised reconstruction under MSE / SSIM loss with Cosine Annealing learning rate.
5. **Evaluation & Visualization:** Quantitative metrics (MSE, PSNR, SSIM) + side-by-side triplet visualizer.
6. **Experimental Study:** Systematic analysis across noise levels and bottleneck dimensions.

## 1. Environment & GPU Acceleration Check

In [ ]:
!nvidia-smi

import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name:     {torch.cuda.get_device_name(0)}")

## Mount Drive and enter the code folder
Upload this `q1/` folder to `MyDrive/as2/q1` first. Checkpoints, results and the dataset are written inside it, so they survive a disconnect.

In [ ]:
from google.colab import drive; drive.mount('/content/drive')
%cd /content/drive/MyDrive/as2/q1

## 2. Install Dependencies

In [ ]:
!pip install -r requirements.txt

## 3. Dataset Preparation & Noise Injection (Tasks 1 & 2)
Download the Berkeley Segmentation Dataset (BSDS500) and verify artificial noise injection.

In [ ]:
!python dataset.py --download --data_dir ./data/BSDS500 --save_sample ./sample_noise.png

from IPython.display import Image, display
display(Image('./sample_noise.png'))

## 4. DAE Architecture Inspection (Task 3)
Inspect the Convolutional Denoising Autoencoder with customizable bottleneck compression.

In [ ]:
!python models.py --bottleneck 32

## 5. Model Training (Task 4)
Train ConvDAE models under both Gaussian noise and Salt-and-Pepper noise.

In [ ]:
# Train on Gaussian Noise (std = 0.15)
!python train.py \
  --data_dir ./data/BSDS500 \
  --noise_type gaussian \
  --noise_param 0.15 \
  --bottleneck 32 \
  --loss mse \
  --epochs 35 \
  --batch_size 16 \
  --lr 0.001

In [ ]:
# Train on Salt-and-Pepper Noise (corruption ratio p = 0.10)
!python train.py \
  --data_dir ./data/BSDS500 \
  --noise_type salt_and_pepper \
  --noise_param 0.10 \
  --bottleneck 32 \
  --loss mse \
  --epochs 35 \
  --batch_size 16 \
  --lr 0.001

## 6. Evaluation & Triplet Visualizations (Task 5)
Evaluate reconstruction quality on the test split using MSE, PSNR, and SSIM.
Visualizations show:
- Row 1: Original clean ground truth
- Row 2: Corrupted noisy input image
- Row 3: Reconstructed denoised output image

In [ ]:
# Evaluate Gaussian DAE on Test Split
!python evaluate.py \
  --checkpoint ./checkpoints/dae_gaussian_p0.15_btn32_mse/best_model.pth \
  --data_dir ./data/BSDS500 \
  --split test \
  --num_samples 5 \
  --save_dir ./results

display(Image('./results/visualization_best_model_gaussian_0.15.png'))

In [ ]:
# Evaluate Salt-and-Pepper DAE on Test Split
!python evaluate.py \
  --checkpoint ./checkpoints/dae_salt_and_pepper_p0.1_btn32_mse/best_model.pth \
  --data_dir ./data/BSDS500 \
  --split test \
  --num_samples 5 \
  --save_dir ./results

display(Image('./results/visualization_best_model_salt_and_pepper_0.1.png'))

## 7. Experimental Study (Task 6)
Analyze reconstruction performance as a function of:
1. **Noise Intensity:** Gaussian ($\sigma \in [0.05, 0.10, 0.20, 0.30]$) and Salt-and-Pepper ($p \in [0.02, 0.05, 0.10, 0.20]$)
2. **Bottleneck Capacity:** Channel widths $\in [8, 16, 32, 64, 128]$

In [ ]:
# Run complete experimental sweep
!python experiments.py \
  --data_dir ./data/BSDS500 \
  --output_dir ./experiments_output \
  --epochs 20 \
  --batch_size 16

### View Experimental Study Curves

In [ ]:
print("Study 1: Performance vs Noise Levels:")
display(Image('./experiments_output/noise_study_curves.png'))

print("Study 2: Performance vs Bottleneck Capacity:")
display(Image('./experiments_output/bottleneck_study_curves.png'))

### View Summary Tables & Report

In [ ]:
import pandas as pd
from IPython.display import Markdown

print("--- Noise Study Results Table ---")
df_noise = pd.read_csv('./experiments_output/noise_study_results.csv')
display(df_noise[['Noise Type', 'Noise Level', 'Input PSNR (dB)', 'Output PSNR (dB)', 'Delta PSNR (dB)', 'Output SSIM']])

print("\n--- Bottleneck Study Results Table ---")
df_btn = pd.read_csv('./experiments_output/bottleneck_study_results.csv')
display(df_btn[['Bottleneck Channels', 'Total Parameters', 'Compression Ratio', 'Output PSNR (dB)', 'Output SSIM', 'Delta PSNR (dB)']])

with open('./experiments_output/experimental_study_report.md', 'r') as f:
    display(Markdown(f.read()))